# Semi-Structured Data Lab: QuickKart Order Documents (JSON)

**Course:** Big Data Analytics — MBA BA, Trim IV
**Business Case:** QuickKart's order data as it actually looks in production — nested, evolving, JSON documents

---

### Why semi-structured data is a different beast

In the previous notebook, QuickKart's orders lived in a clean, flat CSV — every row had exactly the same columns. That's **structured data**, and it's the easiest kind to work with, but it's also not how most real systems actually store order data.

In practice, QuickKart's app backend stores each order as a **JSON document**: one order can contain a *nested list* of items (a customer might buy 5 different products in one order), and not every order document has the same fields — QuickKart added a `promo_code` field to their checkout flow halfway through the year, so older orders simply don't have it.

This is **semi-structured data**: it has *some* organization (keys, nesting, a general shape) but no single rigid schema every record must obey. Think MongoDB documents, API responses, log entries, IoT sensor readings — all semi-structured.

### The business question

> **"What is total revenue by category, and what fraction of orders used a promo code — even though promo codes didn't exist for the first part of the year?"**

This is a realistic BI question that a rigid, fixed-schema system would struggle with cleanly, but semi-structured storage handles naturally via **schema-on-read**: you don't decide the structure when you store the data, you decide it when you query it.


In [ ]:
import json, os, shutil, random
import pandas as pd
import numpy as np

random.seed(7)
np.random.seed(7)

categories = ["Fruits & Vegetables", "Dairy", "Snacks", "Beverages",
              "Household", "Personal Care", "Bakery", "Frozen Foods"]
regions = ["North", "South", "West"]
promo_codes = ["WELCOME10", "FESTIVE20", "FREESHIP", None, None, None]  # None = no promo used

def make_order(order_id, promo_field_exists):
    n_items = random.randint(1, 4)
    items = [
        {
            "category": random.choice(categories),
            "qty": random.randint(1, 3),
            "unit_price": round(random.uniform(30, 400), 2),
        }
        for _ in range(n_items)
    ]
    order = {
        "order_id": order_id,
        "region": random.choice(regions),
        "items": items,   # <-- nested list, no fixed length
        "customer": {"loyalty_tier": random.choice(["Bronze", "Silver", "Gold"])},
    }
    # Schema drift: promo_code field only exists on orders placed after the mid-year feature launch
    if promo_field_exists:
        order["promo_code"] = random.choice(promo_codes)
    return order

# First 4000 orders: NO promo_code field exists at all (feature didn't exist yet)
# Next 4000 orders: promo_code field exists (may still be null if unused)
orders = [make_order(i, promo_field_exists=False) for i in range(1, 4001)] + \
         [make_order(i, promo_field_exists=True) for i in range(4001, 8001)]

print(f"Total orders generated: {len(orders)}")
print("\nExample order WITHOUT promo_code field (early in the year):")
print(json.dumps(orders[0], indent=2))
print("\nExample order WITH promo_code field (after feature launch):")
print(json.dumps(orders[-1], indent=2))


Notice the two example orders printed above **do not have the same set of keys**. The first one has no `promo_code` field at all — not even `null` — it simply doesn't exist. A traditional relational table can't represent that gracefully; every row *must* have every column. JSON documents can.


In [ ]:
# Store the documents the way a real semi-structured store would: as JSON Lines (.jsonl),
# one document per line. We'll split them across 2 "shard" files, similar to how a
# document store spreads collections across multiple physical files/nodes.
if os.path.exists("semistructured_store"):
    shutil.rmtree("semistructured_store")
os.makedirs("semistructured_store", exist_ok=True)

mid = len(orders) // 2
shard_files = {
    "semistructured_store/shard1.jsonl": orders[:mid],
    "semistructured_store/shard2.jsonl": orders[mid:],
}

for path, shard in shard_files.items():
    with open(path, "w") as f:
        for order in shard:
            f.write(json.dumps(order) + "\n")

for path in shard_files:
    size_kb = os.path.getsize(path) / 1024
    print(f"{path}: {size_kb:.1f} KB")


## Schema-on-read: turning JSON into something analyzable

This is the key semi-structured concept. We didn't define any schema before writing the data — we're only deciding *now, at read time* how to interpret it. We'll use `pandas.json_normalize` to flatten the nested `items` list into individual rows, one per item, while keeping the order-level fields attached.


In [ ]:
def load_shard(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

all_orders = load_shard("semistructured_store/shard1.jsonl") + load_shard("semistructured_store/shard2.jsonl")

# Flatten: one row per item, but keep order_id, region, promo_code (if present) attached
rows = []
for order in all_orders:
    for item in order["items"]:
        rows.append({
            "order_id": order["order_id"],
            "region": order["region"],
            "loyalty_tier": order["customer"]["loyalty_tier"],
            "promo_code": order.get("promo_code", "FIELD_NOT_INTRODUCED_YET"),
            "category": item["category"],
            "revenue": item["qty"] * item["unit_price"],
        })

flat = pd.DataFrame(rows)
flat.head(10)


Notice the `.get("promo_code", "FIELD_NOT_INTRODUCED_YET")` — this is doing real work. It's the code equivalent of saying *"if this document doesn't have this field, treat it as this value instead of crashing."* This kind of defensive, field-may-or-may-not-exist coding is the everyday reality of working with semi-structured data, and it's exactly why tools like Hive and Spark support flexible/nested schemas instead of forcing everything into rigid tables upfront.


## MapReduce-Style Analytics on the Flattened Data

Same Map → Shuffle → Reduce pattern as the structured-data notebook, applied here to answer the business question: **revenue by category, and promo code usage.**


In [ ]:
from multiprocessing import Pool

def map_shard_revenue(path):
    """MAP step: compute partial category revenue totals for one shard, independently."""
    shard_orders = load_shard(path)
    partial_rows = []
    for order in shard_orders:
        for item in order["items"]:
            partial_rows.append({"category": item["category"], "revenue": item["qty"] * item["unit_price"]})
    partial_df = pd.DataFrame(partial_rows)
    return partial_df.groupby("category")["revenue"].sum()

shard_paths = list(shard_files.keys())

with Pool(2) as pool:
    partial_results = pool.map(map_shard_revenue, shard_paths)

# SHUFFLE + REDUCE: combine partial sums by category
category_revenue = pd.concat(partial_results).groupby(level=0).sum().sort_values(ascending=False)
category_revenue


In [ ]:
# Promo code adoption analysis — only meaningful for orders where the field exists at all
has_promo_field = flat[flat["promo_code"] != "FIELD_NOT_INTRODUCED_YET"]

promo_usage = has_promo_field["promo_code"].value_counts(dropna=False)
promo_usage_pct = (promo_usage / has_promo_field["order_id"].nunique() * 100).round(1)

print("Promo code usage (only among orders where the feature existed):")
print(promo_usage_pct.astype(str) + "%")

pct_orders_with_field = round(len(has_promo_field["order_id"].unique()) / flat["order_id"].nunique() * 100, 1)
print(f"\n{pct_orders_with_field}% of all orders even had the promo_code field at all.")


### Business insight

We just answered a question that spans a **schema change mid-year** without needing to backfill old records, run a database migration, or throw away historical data. This is precisely why companies with fast-moving product teams (adding fields, deprecating fields, nesting new sub-objects) lean on semi-structured storage rather than forcing every change through a rigid relational schema migration.

### Visualizing category revenue


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
category_revenue.sort_values().plot(kind="barh", ax=ax, color="#2E7D32")
ax.set_xlabel("Total Revenue (₹)")
ax.set_title("QuickKart: Revenue by Category (from Nested JSON Orders)")
plt.tight_layout()
plt.show()


## Recap: Structured vs. Semi-Structured

| | Structured (Notebook 1: CSV) | Semi-Structured (this notebook: JSON) |
|---|---|---|
| Schema | Fixed — every row has every column | Flexible — fields can appear/disappear over time |
| Nesting | None — flat rows only | Supports nested objects/arrays (`items` list) |
| Schema enforcement | On write (must match table structure) | On read (you decide structure when you query) |
| Real-world examples | RDBMS exports, clean CSVs | MongoDB documents, JSON APIs, log files |
| Storage tools | HDFS + Hive tables | HDFS + Hive with nested/JSON SerDe, MongoDB, Spark JSON reader |

### Discussion questions for class

1. What would have broken if we'd tried to store this order data in a single rigid CSV from day one, before QuickKart even had a promo code feature?
2. What's the hidden cost of schema-on-read flexibility? (Hint: think about what happens when *every* query has to handle missing fields defensively, the way we did with `.get()`.)
3. QuickKart's `items` list can have 1 to 4 products per order. How would you represent that in a strict relational table? Is that easier or harder than the JSON version?
